## 수정종가 처리

In [42]:
import pandas as pd
import numpy as np
import random
import os
import pickle
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

import pandas_ta as ta
warnings.filterwarnings("ignore")

In [43]:
# Read in price data
path = "./data/combined_train_4.csv"
train = pd.read_csv(path, parse_dates=True, index_col="일자")
train = train.reset_index()
train

,일자,종목코드,종목명,거래량,시가,고가,저가,종가,기준금리,CPI,실업률,환율,시가총액,BPS,PER,PBR,EPS,DIV,DPS
0,2021-06-01,A060310,3S,166690,2890,2970,2885,2920,0.5,2.3,3.7,1106.979980,1.351128e+11,745.0,0.00,3.92,0.0,0.00,0.0
1,2021-06-01,A095570,AJ네트웍스,63836,5860,5940,5750,5780,0.5,2.3,3.7,1106.979980,2.706329e+11,6089.0,0.00,0.95,0.0,3.63,210.0
2,2021-06-01,A006840,AK홀딩스,103691,35500,35600,34150,34400,0.5,2.3,3.7,1106.979980,4.557161e+11,50471.0,0.00,0.68,0.0,1.16,400.0
3,2021-06-01,A054620,APS,462544,14600,14950,13800,14950,0.5,2.3,3.7,1106.979980,3.048936e+11,8135.0,0.00,1.84,0.0,0.00,0.0
4,2021-06-01,A265520,AP시스템,131987,29150,29150,28800,29050,0.5,2.3,3.7,1106.979980,4.206506e+11,9041.0,16.73,3.21,1736.0,0.41,120.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1071995,2023-07-28,A001080,만호제강,12964,35550,36000,34700,36000,3.5,2.7,2.6,1272.810059,1.494000e+11,79971.0,16.61,0.45,2168.0,0.69,250.0
1071996,2023-07-28,A104700,한국철강,72644,5780,6030,5780,6030,3.5,2.7,2.6,1272.810059,2.559735e+11,20372.0,2.93,0.30,2055.0,4.98,300.0
1071997,2023-07-28,A045100,한양이엔지,59562,16230,16390,15970,16330,3.5,2.7,2.6,1272.810059,2.939400e+11,27595.0,4.04,0.59,4044.0,3.67,600.0
1071998,2023-07-28,A000020,동화약품,86169,9870,10080,9700,9800,3.5,2.7,2.6,1272.810059,2.737284e+11,13165.0,13.32,0.74,736.0,1.84,180.0


In [44]:
train.columns = ['date', 'ticker', 'firm', 'volume', 'open', 'high', 'low', 'close', 'rate', 'cpi', 'un_rate', 'fx', 'cap', 'bps', 'per', 'pbr', 'eps', 'div', 'dps']
df = train.sort_values(by=['ticker', 'date'], ascending=True)

df['adjustTrue'] = 1
df.loc[df['volume'] == 0, 'adjustTrue'] = -1
df = df.sort_values(['ticker','date'], ascending=[True,False])
df = df.reset_index(drop=True)
df.tail()

,date,ticker,firm,volume,open,high,low,close,rate,cpi,un_rate,fx,cap,bps,per,pbr,eps,div,dps,adjustTrue
1071995,2021-06-07,A383800,LX홀딩스,2714980,10550,11150,10500,10800,0.5,2.6,3.7,1109.680054,8.238315e+11,NaN,NaN,NaN,NaN,NaN,NaN,1
1071996,2021-06-04,A383800,LX홀딩스,1737593,10450,10650,10350,10450,0.5,2.6,3.7,1115.439941,7.971332e+11,NaN,NaN,NaN,NaN,NaN,NaN,1
1071997,2021-06-03,A383800,LX홀딩스,2709800,10650,10700,10300,10400,0.5,2.6,3.7,1109.880005,7.933192e+11,NaN,NaN,NaN,NaN,NaN,NaN,1
1071998,2021-06-02,A383800,LX홀딩스,2426922,10700,10850,10600,10700,0.5,2.6,3.7,1107.209961,8.162034e+11,NaN,NaN,NaN,NaN,NaN,NaN,1
1071999,2021-06-01,A383800,LX홀딩스,1879288,11000,11300,10900,11000,0.5,2.3,3.7,1106.979980,8.390876e+11,NaN,NaN,NaN,NaN,NaN,NaN,1


In [45]:
# Convert 'date' to datetime and sort the data by date
data = df
data

,date,ticker,firm,volume,open,high,low,close,rate,cpi,un_rate,fx,cap,bps,per,pbr,eps,div,dps,adjustTrue
0,2023-07-28,A000020,동화약품,86169,9870,10080,9700,9800,3.5,2.7,2.6,1272.810059,2.737284e+11,13165.0,13.32,0.74,736.0,1.84,180.0,1
1,2023-07-27,A000020,동화약품,102981,9260,9870,9260,9870,3.5,2.7,2.6,1275.170044,2.756836e+11,13165.0,13.41,0.75,736.0,1.82,180.0,1
2,2023-07-26,A000020,동화약품,202756,9810,9810,9150,9280,3.5,2.7,2.6,1274.119995,2.592040e+11,13165.0,12.61,0.70,736.0,1.94,180.0,1
3,2023-07-25,A000020,동화약품,168069,10030,10110,9740,9790,3.5,2.7,2.6,1280.530029,2.734491e+11,13165.0,13.30,0.74,736.0,1.84,180.0,1
4,2023-07-24,A000020,동화약품,101983,10300,10300,10000,10130,3.5,2.7,2.6,1285.650024,2.829458e+11,13165.0,13.76,0.77,736.0,1.78,180.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1071995,2021-06-07,A383800,LX홀딩스,2714980,10550,11150,10500,10800,0.5,2.6,3.7,1109.680054,8.238315e+11,NaN,NaN,NaN,NaN,NaN,NaN,1
1071996,2021-06-04,A383800,LX홀딩스,1737593,10450,10650,10350,10450,0.5,2.6,3.7,1115.439941,7.971332e+11,NaN,NaN,NaN,NaN,NaN,NaN,1
1071997,2021-06-03,A383800,LX홀딩스,2709800,10650,10700,10300,10400,0.5,2.6,3.7,1109.880005,7.933192e+11,NaN,NaN,NaN,NaN,NaN,NaN,1
1071998,2021-06-02,A383800,LX홀딩스,2426922,10700,10850,10600,10700,0.5,2.6,3.7,1107.209961,8.162034e+11,NaN,NaN,NaN,NaN,NaN,NaN,1


In [46]:
result = []
ticker_list = data['ticker'].unique()

for ticker in tqdm(ticker_list, leave=True):
    temp = data[data['ticker'] == ticker]
    temp = temp.reset_index(drop=True)
    # Find the index where trading was suspended
    suspension_indices = temp[temp['adjustTrue'] == -1].index

    if len(suspension_indices) == 0: # 거래정지가 없는 경우 패스
        result.append(temp)
        continue
    else:
        for index in tqdm(suspension_indices, leave=True):
            # Get the split ratio from the close price at the suspension date and the open price at the date following the suspension
            close_price_at_suspension = temp.loc[index, 'close']
            try: # 23-05-30에 거래정지인 경우 + 다른 거래정지도 고려
                open_price_after_suspension = temp.loc[index-1, 'open']  # 미래 데이터
            except: # 23-05-30 하루만 거래정지면 for문 탈출
                continue
            split_ratio = close_price_at_suspension / open_price_after_suspension if open_price_after_suspension != 0 else 1
            # Adjust the volume, open, high, low, and close prices for all previous dates (because the data is in descending order)
            # 거래 정지 이후 값은 액면분할을 반영하여 덮어쓰기
            temp.loc[index+1:, ['open', 'high', 'low', 'close']] /= split_ratio
            temp.loc[index+1:, 'volume'] *= split_ratio
        
        # Sort the data in ascending order of date
        # 다시 과거-현재 순으로 재정렬
        temp = temp.sort_values('date', ascending=True)

        # Interpolate zero values in the data using 'pad' method
        # 아직도 0이 남았다 == 거래 정지일이 끝 날짜에 하루밖에 없었다
        # 과거, 미래 값으로 채우기
        temp.replace(0, pd.NA, inplace=True)
        temp.interpolate(method='ffill', inplace=True)
        temp.interpolate(method='bfill', inplace=True)

        # 액면분할 시 최초 거래정지일 기준 변경되지 않은 기준 close값 변경
        try:
            temp.loc[suspension_indices[0], 'close'] = temp.loc[suspension_indices[0] - 1, 'close']
        except:
            pass

        result.append(temp)

result = pd.concat(result, axis=0)

100%|██████████| 2000/2000 [01:26<00:00, 23.07it/s]


In [47]:
stock = result.round(2).copy()
st_price = stock.groupby('firm')['close'].agg('mean').sort_values(ascending=False)
print('변환전\n', st_price[:5])

변환전
 firm
태광산업        896792.910448
LG생활건강      892169.776119
삼성바이오로직스    831287.313433
LG화학        674034.514925
삼성SDI       660470.149254
Name: close, dtype: float64


In [48]:
for i, a in enumerate(list(st_price.index)):
    stock.loc[stock['firm'] == a, 'firm'] = i
st_price = result.groupby('firm')['close'].agg('mean').sort_values(ascending=False)
print('변환후\n', st_price[:5])

변환후
 firm
태광산업        896792.910448
LG생활건강      892169.776119
삼성바이오로직스    831287.313433
LG화학        674034.514925
삼성SDI       660470.149254
Name: close, dtype: float64


In [53]:
stock['firm'] = stock['firm'].astype('category').cat.codes
stock

,date,ticker,firm,volume,open,high,low,close,rate,cpi,un_rate,fx,cap,bps,per,pbr,eps,div,dps,adjustTrue
0,2023-07-28,A000020,826,86169.0,9870.0,10080.0,9700.0,9800.0,3.5,2.7,2.6,1272.81,2.737284e+11,13165.0,13.32,0.74,736.0,1.84,180.0,1
1,2023-07-27,A000020,826,102981.0,9260.0,9870.0,9260.0,9870.0,3.5,2.7,2.6,1275.17,2.756836e+11,13165.0,13.41,0.75,736.0,1.82,180.0,1
2,2023-07-26,A000020,826,202756.0,9810.0,9810.0,9150.0,9280.0,3.5,2.7,2.6,1274.12,2.592040e+11,13165.0,12.61,0.70,736.0,1.94,180.0,1
3,2023-07-25,A000020,826,168069.0,10030.0,10110.0,9740.0,9790.0,3.5,2.7,2.6,1280.53,2.734491e+11,13165.0,13.30,0.74,736.0,1.84,180.0,1
4,2023-07-24,A000020,826,101983.0,10300.0,10300.0,10000.0,10130.0,3.5,2.7,2.6,1285.65,2.829458e+11,13165.0,13.76,0.77,736.0,1.78,180.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
531,2021-06-07,A383800,989,2714980.0,10550.0,11150.0,10500.0,10800.0,0.5,2.6,3.7,1109.68,8.238315e+11,NaN,NaN,NaN,NaN,NaN,NaN,1
532,2021-06-04,A383800,989,1737593.0,10450.0,10650.0,10350.0,10450.0,0.5,2.6,3.7,1115.44,7.971332e+11,NaN,NaN,NaN,NaN,NaN,NaN,1
533,2021-06-03,A383800,989,2709800.0,10650.0,10700.0,10300.0,10400.0,0.5,2.6,3.7,1109.88,7.933192e+11,NaN,NaN,NaN,NaN,NaN,NaN,1
534,2021-06-02,A383800,989,2426922.0,10700.0,10850.0,10600.0,10700.0,0.5,2.6,3.7,1107.21,8.162034e+11,NaN,NaN,NaN,NaN,NaN,NaN,1


In [54]:
# 데이터 재배열
stock1 = stock
stock1 = stock1.sort_values(['ticker', 'date'])
stock1.head()

,date,ticker,firm,volume,open,high,low,close,rate,cpi,un_rate,fx,cap,bps,per,pbr,eps,div,dps,adjustTrue
535,2021-06-01,A000020,826,114966.0,14700.0,14700.0,14450.0,14600.0,0.5,2.3,3.7,1106.98,4.077995e+11,11860.0,14.13,1.23,1033.0,1.23,180.0,1
534,2021-06-02,A000020,826,109559.0,14700.0,14700.0,14450.0,14500.0,0.5,2.6,3.7,1107.21,4.050063e+11,11860.0,14.04,1.22,1033.0,1.24,180.0,1
533,2021-06-03,A000020,826,96158.0,14550.0,14650.0,14450.0,14600.0,0.5,2.6,3.7,1109.88,4.077995e+11,11860.0,14.13,1.23,1033.0,1.23,180.0,1
532,2021-06-04,A000020,826,133900.0,14600.0,14800.0,14550.0,14700.0,0.5,2.6,3.7,1115.44,4.105926e+11,11860.0,14.23,1.24,1033.0,1.22,180.0,1
531,2021-06-07,A000020,826,511140.0,14800.0,15550.0,14750.0,15150.0,0.5,2.6,3.7,1109.68,4.231618e+11,11860.0,14.67,1.28,1033.0,1.19,180.0,1


In [55]:
stock1.to_csv("./data/train_adj.csv")

---

## 기술적 분석

In [56]:
def calculate_technical_indicators(df: pd.DataFrame) -> pd.DataFrame:
    # Moving Averages
    df.ta.sma(close='Close', length=10, append=True)
    df.ta.sma(close='Close', length=20, append=True)
    df.ta.sma(close='Close', length=50, append=True)
    df.ta.sma(close='Close', length=100, append=True)
    df.ta.sma(close='Close', length=200, append=True)

    df.ta.ema(close='Close', length=10, append=True)
    df.ta.ema(close='Close', length=20, append=True)
    df.ta.ema(close='Close', length=50, append=True)
    df.ta.ema(close='Close', length=100, append=True)
    df.ta.ema(close='Close', length=200, append=True)

    # Momentum Indicators
    df.ta.rsi(close='Close', length=14, append=True)
    df.ta.macd(close='Close', fast=12, slow=26, signal=9, append=True)
    df.ta.stoch(close='Close', append=True)
    df.ta.roc(close='Close', append=True)
    # MACD는 단기 EMA (12일)가 장기 EMA (26일)에 비해 얼마나 빠르게 움직이는지를 보여줍니다.

    # Volume Indicators
    df.ta.vp(close='Close', volume='Volume', append=True)
    df.ta.obv(close='Close', volume='Volume', append=True)

    # Volatility Indicators
    df.ta.atr(close='Close', append=True)
    df.ta.bbands(close='Close', append=True)

    # Trend Strength Indicators
    df.ta.adx(close='Close', append=True)

    df.ta.efi(length=13, append=True) # Elder's Force Index (EFI): 알렉산더 엘더가 개발한 이 지표는 가격의 변동성과 거래량을 결합하여 주식의 '힘'을 측정합니다.
    df.ta.kama(length=10, append=True) # Kaufman's Adaptive Moving Average (KAMA): 이 지표는 변동성을 고려하여 보다 유연한 이동 평균을 제공합니다.
    df.ta.mfi(high='High', low='Low', close='Close', volume='Volume', length=14, append=True) # Money Flow Index (MFI): 이 지표는 가격과 거래량을 결합하여 주식이 과매수 또는 과매도 상태인지 판단합니다.
    df.ta.vortex(high='High', low='Low', close='Close', length=14, append=True) # Vortex Indicator (VI): 이 지표는 최근 가격의 상승과 하락을 추적하여 상승 추세와 하락 추세를 식별합니다.

    return df


In [57]:
train = pd.read_csv("./data/train_adj.csv")
train = train.drop('Unnamed: 0', axis=1)
train = train.sort_values(['ticker', 'date'], ascending=True)
train.set_index('date', inplace=True)
train

,ticker,firm,volume,open,high,low,close,rate,cpi,un_rate,fx,cap,bps,per,pbr,eps,div,dps,adjustTrue
date,,,,,,,,,,,,,,,,,,,
2021-06-01,A000020,826,114966.0,14700.0,14700.0,14450.0,14600.0,0.5,2.3,3.7,1106.98,4.077995e+11,11860.0,14.13,1.23,1033.0,1.23,180.0,1
2021-06-02,A000020,826,109559.0,14700.0,14700.0,14450.0,14500.0,0.5,2.6,3.7,1107.21,4.050063e+11,11860.0,14.04,1.22,1033.0,1.24,180.0,1
2021-06-03,A000020,826,96158.0,14550.0,14650.0,14450.0,14600.0,0.5,2.6,3.7,1109.88,4.077995e+11,11860.0,14.13,1.23,1033.0,1.23,180.0,1
2021-06-04,A000020,826,133900.0,14600.0,14800.0,14550.0,14700.0,0.5,2.6,3.7,1115.44,4.105926e+11,11860.0,14.23,1.24,1033.0,1.22,180.0,1
2021-06-07,A000020,826,511140.0,14800.0,15550.0,14750.0,15150.0,0.5,2.6,3.7,1109.68,4.231618e+11,11860.0,14.67,1.28,1033.0,1.19,180.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-07-24,A383800,989,232782.0,7850.0,8000.0,7710.0,7800.0,3.5,2.7,2.6,1285.65,5.949894e+11,21133.0,3.56,0.37,2190.0,3.97,310.0,1
2023-07-25,A383800,989,3965325.0,7850.0,8960.0,7700.0,8140.0,3.5,2.7,2.6,1280.53,6.209248e+11,21133.0,3.72,0.39,2190.0,3.81,310.0,1
2023-07-26,A383800,989,2042248.0,8150.0,8850.0,7780.0,7980.0,3.5,2.7,2.6,1274.12,6.087199e+11,21133.0,3.64,0.38,2190.0,3.88,310.0,1


In [58]:
ticker_list = train['ticker'].unique()
data_frames = []  # store DataFrames here

for ticker in tqdm(ticker_list):
    temp = calculate_technical_indicators(train[train['ticker'] == ticker])
    data_frames.append(temp)

# concat all at once
data = pd.concat(data_frames, axis=0)
data.head()

100%|██████████| 2000/2000 [01:48<00:00, 18.42it/s]


,ticker,firm,volume,open,high,low,close,rate,cpi,un_rate,...,BBB_5_2.0,BBP_5_2.0,ADX_14,DMP_14,DMN_14,EFI_13,KAMA_10_2_30,MFI_14,VTXP_14,VTXM_14
date,,,,,,,,,,,,,,,,,,,,,
2021-06-01,A000020,826,114966.0,14700.0,14700.0,14450.0,14600.0,0.5,2.3,3.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-06-02,A000020,826,109559.0,14700.0,14700.0,14450.0,14500.0,0.5,2.6,3.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-06-03,A000020,826,96158.0,14550.0,14650.0,14450.0,14600.0,0.5,2.6,3.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-06-04,A000020,826,133900.0,14600.0,14800.0,14550.0,14700.0,0.5,2.6,3.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-06-07,A000020,826,511140.0,14800.0,15550.0,14750.0,15150.0,0.5,2.6,3.7,...,6.224622,0.980537,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [59]:
data_ta = data.dropna(axis=1, how='all')
data_ta = data_ta.dropna(axis=0)
ticker_list = data_ta['ticker'].unique()
print("ticker_list 길이 : ",  len(ticker_list))

ticker_list 길이 :  1844


In [60]:
data_ta.to_pickle("./data/train_adj_tactical.pkl")

---

## Train / Test 분리

In [61]:
import pandas as pd
import numpy as np
import random
import os
import pickle
from tqdm import tqdm

import pandas_ta as ta

import warnings
warnings.filterwarnings("ignore")

In [62]:
data_ta = pd.read_pickle("./data/train_adj_tactical.pkl")
data_ta = data_ta.reset_index()

In [63]:
data_ta = data_ta.sort_values(['ticker', 'date'])
data_ta2 = data_ta.copy()
data_ta2['target'] = data_ta.groupby('firm')['close'].shift(-15)

In [64]:
train = data_ta2[~data_ta2['target'].isna()]
test = data_ta2[data_ta2['target'].isna()]

In [65]:
train['pct_change'] = ((train['target'] - train['close']) / train['close'] * 100).round(5)

def up_or_down(df):
    conditions = [
        (df['pct_change'] > 4), 
        (df['pct_change'] > 0),
        (df['pct_change'] < 0)
    ]
    choices = [2, 1, 0]
    df['class_target'] = np.select(conditions, choices, default=0)

    return df

train = up_or_down(train)
train.head()

,date,ticker,firm,volume,open,high,low,close,rate,cpi,...,DMP_14,DMN_14,EFI_13,KAMA_10_2_30,MFI_14,VTXP_14,VTXM_14,target,pct_change,class_target
0,2022-03-23,A000020,826,396150.0,13800.0,14100.0,13600.0,13650.0,1.25,3.7,...,30.358104,13.516124,2.075979e+08,13031.963084,74.680545,1.109091,0.842424,12950.0,-5.12821,0
1,2022-03-24,A000020,826,164839.0,13600.0,13700.0,13500.0,13600.0,1.25,3.7,...,29.627900,14.393671,1.767636e+08,13056.980792,72.143290,1.086957,0.913043,13200.0,-2.94118,0
2,2022-03-25,A000020,826,248995.0,13700.0,13950.0,13500.0,13900.0,1.25,3.7,...,31.055802,13.600975,1.621829e+08,13116.285955,73.136564,1.078788,0.872727,13250.0,-4.67626,0
3,2022-03-28,A000020,826,160036.0,13900.0,13900.0,13600.0,13750.0,1.25,3.7,...,29.874583,13.083657,1.355845e+08,13175.382776,70.754830,1.092593,0.901235,13200.0,-4.00000,0
4,2022-03-29,A000020,826,160334.0,13850.0,14000.0,13650.0,13750.0,1.25,3.7,...,29.815148,12.486931,1.162153e+08,13215.806253,72.694644,1.118750,0.868750,13200.0,-4.00000,0


In [66]:
train.to_pickle("./data/train_완료.pkl")
test.to_pickle("./data/prediction_완료.pkl")